In [ ]:
import pandas as pd
%load_ext autoreload
%autoreload 2

from pathlib import Path
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import napari
import colorcet as cc

import dnt

spots_directory = Path(r"C:\Tracking\BlastodermAnalysis\data\spots")
embryo_overview = pd.read_excel(spots_directory / "overview.xlsx", sheet_name="Sheet1")
save_path = Path(r"C:\Tracking\BlastodermAnalysis\figures\figure_1")


embryo_overview = embryo_overview[embryo_overview["good"]]
included = embryo_overview["Embryo"].astype(str).tolist()
condition_map = {
    str(embryo): condition for embryo, condition in zip(embryo_overview["Embryo"], embryo_overview["condition"])
}
print(condition_map)

dnt.set_plot_style()
spots_dfs, stems = dnt.load_spots_data(spots_directory, included)

print(stems)

df = spots_dfs[0]
cycles = [10, 11, 12, 13, 14]

print(df.columns)

greens = ["#143601","#1a4301","#245501","#538d22","#73a942","#aad576"][::-1]
blues = ["#012a4a","#01497c","#2a6f97","#468faf","#89c2d9"][::-1]
reds = ["#641220","#85182a","#a71e34","#bd1f36", "#da1e37"][::-1]
oranges = ["#fbba72","#ca5310","#bb4d00","#8f250c","#691e06"]
condition_pal_map = {
    "wt": blues,
    "bcd": reds,
    "trk": greens,
}
condition_main_colors = {
    c: cmap[2] for c, cmap in condition_pal_map.items()
}
n = 3
condition_pal = {
    "wt": blues[n],
    "bcd": reds[2],
    "trk": greens[2],
}

## Density and surface area calculations

In [ ]:
import dnt
import pandas as pd
from collections import defaultdict

all_surface_areas = defaultdict(dict)
all_mmfs = {}

for k in range(len(spots_dfs)):
    stem = stems[k]
    df = spots_dfs[k]

    min_mvmt_frames, times = dnt.find_stationary_timepoints(df)
    all_mmfs[stem] = min_mvmt_frames

    points = df[df["frame"] == min_mvmt_frames[-1]][["x", "y", "z"]].values
    aps = df[df["frame"] == min_mvmt_frames[-1]]["AP"].values
    mesh = dnt.mesh_from_points(points)

    ap_vals = np.linspace(0.02, 0.98, 25)
    y_max = points[:, 1].max()
    y_min = points[:, 1].min()
    y_vals = y_min + ap_vals * (y_max - y_min)

    try:
        surface_areas = dnt.calculate_surface_area_along_axis(mesh, y_vals)
    except Exception as e:
        mesh = dnt.smoothed_mesh_from_points(points)
        surface_areas = dnt.calculate_surface_area_along_axis(mesh, y_vals)
        # print(f"Error calculating surface areas for embryo {k}: {e}")
        # continue

    # flip order of surface areas if y_max corresponds to AP=0
    if aps[np.argmax(points[:, 1])] < aps[np.argmin(points[:, 1])]:
        surface_areas = surface_areas[::-1]

    total_surface_area = surface_areas.sum()

    ap_bounds = [(l, r) for l, r in zip([0.0, *ap_vals], [*ap_vals, 1.0])]
    centers = [(l + r)/2 for l, r in ap_bounds]

    all_surface_areas[stem]["centers"] = centers
    all_surface_areas[stem]["areas"] = surface_areas
    all_surface_areas[stem]["bounds"] = ap_bounds

    all_surface_areas[stem] = pd.DataFrame(all_surface_areas[stem])

    spots_dfs[k]["AP_bin"] = pd.cut(
        spots_dfs[k]["AP"], bins=[0.0, *ap_vals, 1.0], labels=centers
    )

In [ ]:
for stem, surface_area_df in all_surface_areas.items():
    surface_area_df["area_normed"] = surface_area_df["areas"] / surface_area_df["areas"].sum()
    surface_area_df["area_normed"] = surface_area_df["area_normed"].astype(float)
    surface_area_df["center_copy"] = surface_area_df["centers"]
    surface_area_df.set_index("center_copy", inplace=True)

cycle_relative_densities = defaultdict(list)
intermediate_densities = defaultdict(list)

for k, spots_df in enumerate(spots_dfs):
    stem = stems[k]

    """
    Density related metrics
    """
    spots_df["nucleus_weight"] = 1 / spots_df["frame"].map(spots_df.groupby("frame").size())
    spots_df["local_area"] = np.array(spots_df["AP_bin"].map(all_surface_areas[stems[k]]["area_normed"]))
    spots_df["AP_bin"] = np.array(spots_df["AP_bin"].astype(float))
    spots_df["nucleus_rel_density"] = spots_df["nucleus_weight"] / spots_df["local_area"]

    """
    Displacement related metrics
    """
    spots_df["time_since_nc10"] = np.abs(spots_df["frame"] - all_mmfs[stem][0])
    idx = spots_df.groupby("track_id")["time_since_nc10"].idxmin()
    result = spots_df.loc[idx].set_index("track_id")["AP"]
    spots_df["track_AP_init"] = spots_df["track_id"].map(result)
    spots_df["displacement_from_start"] = spots_df["AP"] - spots_df["track_AP_init"]
    last_frame = all_mmfs[stem][-1]
    spots_df["final_displacement"] = spots_df["track_id"].map(spots_df[spots_df["frame"] == last_frame].groupby("track_id")["AP"].mean()) - spots_df["AP"]

    """
    Get density and displacement at at min movement frames for each cycle
    """
    min_mvmt_frames = all_mmfs[stem]
    for cycle, frame in zip(cycles, min_mvmt_frames):
        cycle_df = spots_df[spots_df["frame"] == frame].copy()

        positions = cycle_df.groupby("AP_bin")["nucleus_rel_density"].mean().index.values
        densities = cycle_df.groupby("AP_bin")["nucleus_rel_density"].sum().values
        displacements = cycle_df.groupby("AP_bin")["displacement_from_start"].mean().values
        final_displacements = cycle_df.groupby("AP_bin")["final_displacement"].mean().values
        surface_areas = cycle_df.groupby("AP_bin")["local_area"].mean().values

        cycle_relative_densities["positions"].extend(positions)
        cycle_relative_densities["densities"].extend(densities)
        cycle_relative_densities["cycle"].extend([cycle for _ in range(len(positions))])
        cycle_relative_densities["condition"].extend([condition_map[stems[k][:-6]] for _ in range(len(positions))])
        cycle_relative_densities["source"].extend([k for _ in range(len(positions))])
        cycle_relative_densities["avg_displacement"].extend(displacements)
        cycle_relative_densities["avg_final_displacement"].extend(final_displacements)
        cycle_relative_densities["surface_area"].extend(surface_areas)

        if cycle == 14:
            intermediate_densities["positions"].extend(positions)
            intermediate_densities["densities"].extend(densities)
            intermediate_densities["cycle"].extend([cycle for _ in range(len(positions))])
            intermediate_densities["condition"].extend([condition_map[stems[k][:-6]] for _ in range(len(positions))])
            intermediate_densities["source"].extend([k for _ in range(len(positions))])
            continue

        for intermediate_fraction in np.arange(0, 1, 0.25):
            cycle_frame = all_mmfs[stem][cycle - 10]
            next_cycle_frame = all_mmfs[stem][cycle - 9]
            intermediate_frame = int(cycle_frame + intermediate_fraction * (next_cycle_frame - cycle_frame))
            intermediate_df = spots_df[spots_df["frame"] == intermediate_frame].copy()

            positions = intermediate_df.groupby("AP_bin")["nucleus_rel_density"].mean().index.values
            densities = intermediate_df.groupby("AP_bin")["nucleus_rel_density"].sum().values

            intermediate_densities["positions"].extend(positions)
            intermediate_densities["densities"].extend(densities)
            intermediate_densities["cycle"].extend([cycle + intermediate_fraction for _ in range(len(positions))])
            intermediate_densities["condition"].extend([condition_map[stems[k][:-6]] for _ in range(len(positions))])
            intermediate_densities["source"].extend([k for _ in range(len(positions))])

cycle_relative_densities = pd.DataFrame(cycle_relative_densities)
intermediate_densities = pd.DataFrame(intermediate_densities)
print(cycle_relative_densities.head())

In [ ]:
def get_plotting_df(relative_densities_df):
    # remove most anterior and posterior positions (pole cells)
    plotting_df = relative_densities_df.query("positions > 0.01 and positions < 0.96").copy()

    # apply rolling mean for each density profile
    plotting_df["densities_local_smooth"] = (plotting_df.groupby(["cycle", "source"])["densities"]
                                                 .transform(lambda x: x.rolling(5, center=True, min_periods=1).mean()))

    # normalize by the average density in the middle region (0.4-0.6) for each cycle and source
    normalizing_constants = plotting_df[plotting_df["positions"].between(0.4, 0.6)].groupby(
        ["cycle", "source"])["densities_local_smooth"].mean().reset_index()

    # divide by the normalizing constant
    plotting_df["densities_local_smooth_normed"] = plotting_df.apply(
        lambda row: row["densities_local_smooth"] /
                    normalizing_constants[
                        (normalizing_constants["cycle"] == row["cycle"]) &
                        (normalizing_constants["source"] == row["source"])
                    ]["densities_local_smooth"].values[0], axis=1)

    return plotting_df

def plot_compare_conditions_at_cycle(relative_densities_df, cycle, conditions, ax, legend=False):
    plotting_df = get_plotting_df(relative_densities_df)

    filtered_df = plotting_df.query("cycle == @cycle and condition in @conditions")

    for k in filtered_df["source"].unique():
        source_subset = filtered_df[filtered_df["source"] == k].copy()
        sns.lineplot(source_subset, x="positions", y="densities_local_smooth", hue="condition", palette=condition_pal, errorbar=None, alpha=0.3, legend=False, ax=ax)

    sns.lineplot(filtered_df, x="positions", y="densities_local_smooth", hue="condition", palette=condition_pal, lw=3, legend=legend, ax=ax, errorbar=None)


def plot_compare_cycles_at_condition(relative_densities_df, condition, cycles, pal, legend=False, style="-"):
    plotting_df = get_plotting_df(relative_densities_df)

    filtered_df = plotting_df.query("condition == @condition and cycle in @cycles")


    # for k in filtered_df["source"].unique():
    #     source_subset = filtered_df[filtered_df["source"] == k].copy()
    #     sns.lineplot(source_subset, x="positions", y="densities_local_smooth", hue="cycle", palette=cycle_pal, errorbar=None, alpha=0.3, legend=False, ax=ax)

    sns.lineplot(filtered_df, x="positions", y="densities_local_smooth", hue="cycle", palette=pal, lw=3, legend=legend, ax=ax, linestyle=style, errorbar=("sd", 1))

fig, axes = plt.subplots(1, 5, figsize=(12, 1.5), sharey=True)
for cycle, ax in zip([10, 11, 12, 13, 14], axes):
    # fig, ax = plt.subplots(1, 1, figsize=(4, 2.5))
    conditions = ["wt", "trk"]
    plot_compare_conditions_at_cycle(cycle_relative_densities, cycle, conditions, ax, legend=False)
    # plt.legend(loc="lower right")
    ax.set_xlabel("AP position")
    ax.set_xticks([0, 0.5, 1.0])
    # ax.set_ylabel("Relative density")
    # ax.set_ylim(0.5, 1.2)
    ax.set_title(f"Cycle {cycle}")
    ax.spines[["top", "right"]].set_visible(False)
axes[0].set_ylim(0.55, 1.3)
axes[0].set_ylabel("Relative Density")
plt.savefig(save_path / f"nc_{cycle}_density_{"".join(conditions)}_comparison.png", dpi=300, bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(9, 1.5), sharey=True)
for cycle, ax in zip([10,  12, 14], axes):
    # fig, ax = plt.subplots(1, 1, figsize=(4, 2.5))
    conditions = ["wt", "trk"]
    plot_compare_conditions_at_cycle(cycle_relative_densities, cycle, conditions, ax, legend=False)
    # plt.legend(loc="lower right")
    ax.set_xlabel("AP position")
    ax.set_xticks([0, 0.5, 1.0])
    # ax.set_ylabel("Relative density")
    # ax.set_ylim(0.5, 1.2)
    ax.set_title(f"Cycle {cycle}")
    ax.spines[["top", "right"]].set_visible(False)
axes[0].set_ylim(0.55, 1.3)
axes[0].set_ylabel("Relative Density")
plt.savefig(save_path / f"nc_{cycle}_density_{"".join(conditions)}_3cycle_comparison.png", dpi=300, bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(2, 1, figsize=(3.5, 5), sharex=True, sharey=True)
for cycle, ax in zip([10, 14], axes):
    conditions = ["wt", "trk"]
    plot_compare_conditions_at_cycle(cycle_relative_densities, cycle, conditions, ax, legend=False)
    ax.set_ylabel("Relative density")
    ax.spines[["top", "right"]].set_visible(False)

axes[1].set_xlabel("AP position")
plt.savefig(save_path / f"nc_10-14_density_{"".join(conditions)}_comparison.png", dpi=300, bbox_inches="tight")
plt.show()


for cycle in [11, 12, 13, 14]:
    fig, ax = plt.subplots(1, 1, figsize=(3.76, 2.46))
    conditions = ["wt", "bcd"]
    plot_compare_conditions_at_cycle(cycle_relative_densities, cycle, conditions, ax)
    plt.xlabel("AP position")
    plt.ylabel("Relative density")
    # plt.title(f"Cycle {cycle}")
    ax.spines[["top", "right"]].set_visible(False)
    plt.ylim(0.6, 1.2)
    plt.savefig(save_path / f"nc_{cycle}_density_{"".join(conditions)}_comparison.png", dpi=300, bbox_inches="tight")
    plt.show()



fig, ax = plt.subplots(1, 1, figsize=(4., 2.5))
cycles = [10, 14]
condition = "wt"
cycle_pal = {c: color for c, color in zip(cycles, condition_pal_map[condition][::4])}
plot_compare_cycles_at_condition(cycle_relative_densities.query("source != 3"), condition, cycles, cycle_pal)
plt.xlabel("AP position")
plt.ylabel("Relative density")
plt.title(f"Density along AP axis")
ax.spines[["top", "right"]].set_visible(False)
plt.savefig(save_path / f"{condition}_cycle_{cycles[0]}_vs_{cycles[1]}_density_comparison.png", dpi=300, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(1, 1, figsize=(4., 2.5))
cycles = [10, 14]
condition = "trk"
cycle_pal = {c: color for c, color in zip(cycles, condition_pal_map[condition][::4])}
plot_compare_cycles_at_condition(cycle_relative_densities.query("source != 9"), condition, cycles, cycle_pal)
plt.xlabel("AP position")
plt.ylabel("Relative density")
plt.title(f"Density along AP axis")
ax.spines[["top", "right"]].set_visible(False)
plt.savefig(save_path / f"{condition}_cycle_{cycles[0]}_vs_{cycles[1]}_density_comparison.png", dpi=300, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(1, 1, figsize=(8, 4))
cycles = np.arange(10, 14.25, 1.0)
condition = "trk"
plot_compare_cycles_at_condition(intermediate_densities, condition, cycles, "Spectral", legend=True)
plt.xlabel("AP position")
plt.ylabel("Relative density")
plt.title(f"Density along AP axis")
ax.spines[["top", "right"]].set_visible(False)
# plt.savefig(save_path / f"{condition}_cycle_{cycles[0]}_vs_{cycles[1]}_density_comparison.png", dpi=300, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(1, 1, figsize=(8, 4))
cycles = [10, 14]
condition = "trk"
plot_compare_cycles_at_condition(intermediate_densities, "trk", cycles, "Spectral", legend=True, style="--")
plot_compare_cycles_at_condition(intermediate_densities, "wt", cycles, "Spectral", legend=False)

plt.xlabel("AP position")
plt.ylabel("Relative density")
plt.title(f"Condition: {condition}")
ax.spines[["top", "right"]].set_visible(False)
# plt.savefig(save_path / f"{condition}_cycle_{cycles[0]}_vs_{cycles[1]}_density_comparison.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
sns.lineplot(cycle_relative_densities.query("cycle == 14"), x="positions", y="surface_area", hue="condition")
plt.xlim(0.05, 0.95)

In [ ]:
k = 0

fig, ax = plt.subplots(1, 1, figsize=(4, 2.5))
df = spots_dfs[k].copy()
first_frame = all_mmfs[stems[0]][0]
df = df[df["frame"] >= first_frame].copy()
df["initial_position"] = df.groupby("track_id")["AP"].transform("first")
df["displacement_from_start"] = df["AP"] - df["initial_position"]
df["initial_cycle"] = df.groupby("track_id")["cycle"].transform("first")

df = df[df["frame"] == all_mmfs[stems[0]][-1]].copy()
df = df.query("initial_cycle == 10")
t = df.groupby("track_id")[["AP", "displacement_from_start", "initial_cycle"]].mean().reset_index()

sns.scatterplot(t, x="AP", y="displacement_from_start", hue="initial_cycle", palette=dnt.palettes.nc, edgecolor="k", legend=False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.xlabel("AP Position")
plt.ylabel("Displacement along AP axis")
plt.axhline(0, linestyle="--", linewidth=2, color="k")
plt.savefig(save_path / f"{stems[k]}_dislacement.png", dpi=300, bbox_inches="tight")
plt.show()
#
# print(stems[k])
# plotting_df = get_plotting_df(cycle_relative_densities).query("source == @k and cycle in [12]")
# sns.lineplot(plotting_df, x="positions", y="densities_local_smooth", hue="cycle", lw=3, legend=True)
# plt.show()


In [ ]:
for k in range(4):
    df = spots_dfs[k]
    print(stems[k])

    for frame in all_mmfs[stems[k]]:
        print(len(df.query("frame == @frame")), end = " & ")

In [ ]:
location_subset = cycle_relative_densities[cycle_relative_densities["positions"].between(0.00, 1.0)].copy()
condition_subset = location_subset[~location_subset["source"].isin([3, 9])]
condition_subset = condition_subset[condition_subset["condition"].isin(["wt", "trk"])]

for cycle in [12, 14]:

    fig, ax = plt.subplots(1, 1, figsize=(3, 2))

    for k in range(len(spots_dfs)):
        if k not in condition_subset["source"].unique():
            continue

        source_subset = condition_subset[condition_subset["source"] == k].copy()

        # sns.lineplot(source_subset[source_subset["cycle"] == cycle], x="positions", y="avg_displacement", hue="condition", palette=condition_main_colors, errorbar=None, alpha=0.3, legend=False)

    print(condition_subset)
    gg = condition_subset.query("cycle == @cycle").groupby(["source", "condition", "positions"])["avg_displacement"].mean()
    print(gg)
    last_cycle = cycle - 2
    gg2 = condition_subset.query("cycle == @last_cycle").groupby(["source", "condition", "positions"])["avg_displacement"].mean()

    avg_displacment = (gg - gg2).reset_index()
    print(avg_displacment.reset_index())
    avg_displacment["avg_displacement"] = avg_displacment["avg_displacement"]*100

    # for src in avg_displacment["source"]:
    #
    #     sns.lineplot(avg_displacment.query("source == @src"), x="positions", y="avg_displacement", hue="condition", palette=condition_main_colors, lw=3, legend=False)

    sns.lineplot(avg_displacment, x="positions", y="avg_displacement", hue="condition", palette=condition_main_colors, lw=3, legend=False)

    # sns.lineplot(condition_subset[condition_subset["cycle"] == cycle], x="positions", y="avg_displacement", hue="condition", palette=condition_main_colors, lw=3, legend=False, errorbar=None)

    plt.axhline(0, linestyle="--", linewidth=2, color="k")
    plt.ylim(-5, 10)
    ax.spines[["top", "right"]].set_visible(False)
    plt.xlabel("AP position")
    plt.ylabel("")
    # plt.title(f"Cycle {cycle}")
    plt.savefig(save_path / f"nc_{cycle}_displacement_comparison.png", dpi=300, bbox_inches="tight")
    plt.show()


In [ ]:
k = stems.index("20250131_spots")

k = 0
print(stems[k])
df = spots_dfs[k]

frame = all_mmfs[stems[k]][0]
points = df[df["frame"] == frame][["x", "y", "z", "radius"]]
points.to_csv(save_path / f"{stems[k]}_frame{frame}_nuclei.csv")

frame_df = df[df["frame"] == all_mmfs[stems[k]][-1]]
good_track_ids = frame_df.groupby("track_id").size()[frame_df.groupby("track_id").size() > 8].index
frame_df = frame_df[frame_df["track_id"].isin(good_track_ids)]
print(frame_df["track_id"].nunique())

condition = condition_map[stems[k][:-6]]
fig, ax = plt.subplots(1, 1, figsize=(2.5, 2))
sns.scatterplot(frame_df.groupby("track_id")[["AP", "displacement_from_start"]].mean(), x="AP", y="displacement_from_start", color=condition_pal_map[condition][2], edgecolor="k")
plt.xlabel("AP position")
plt.ylabel("Displacement along AP axis")
plt.axhline(0, color="k", linestyle="--", lw=3)
ax.spines[["top", "right"]].set_visible(False)
plt.savefig(save_path / f"{stems[k]}_displacement_scatter.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
for k in range(len(spots_dfs)):
    print(stems[k])
    df = spots_dfs[k]
    df = df[df["frame"] == all_mmfs[stems[k]][-1]]
    good_track_ids = df.groupby("track_id").size()[df.groupby("track_id").size() > 8].index
    df = df[df["track_id"].isin(good_track_ids)]
    print(df.groupby("track_id")["displacement_from_start"].mean().mean())

In [ ]:
for cycle in [11, 12, 13]:
    for k, df in enumerate(spots_dfs):
        if stems[k] == "20250705_spots":
            continue

        df = df[df["cycle"] == cycle].copy()
        df = df[df["distance"] < 3]
        df = df[df["pseudotime"] > 0.04]
        df = df[df["AP"].between(0.4, 0.6)]
        x = df["time_since_nc11"] - df["time_since_nc11"].min()
        y = df["dAP"]

        condition = condition_map[stems[k][:-6]]

        if condition != "wt":
            continue
        # if condition == "trk":
        #     continue

        sns.lineplot(x=x, y=y, color=condition_main_colors[condition], alpha=1, errorbar=None, label=f"{stems[k][:-6]}", legend=False, lw=2)

    plt.title(f"NC {cycle}")
    plt.xlabel("Time since division (minutes)")
    plt.ylabel("dAP/dt (embryo lengths per minute)")
    plt.savefig(save_path / f"nc_{cycle}_dAP_realtime.png", dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
for cycle in [11, 12, 13]:
    for k, df in enumerate(spots_dfs):
        df = df[df["cycle"] == cycle].copy()
        df = df[df["distance"] < 3]
        t = df.groupby("tracklet_id")
        start_times = t["time_since_nc11"].min()
        end_times = t["time_since_nc11"].max()
        tracklet_ap_bins = t["AP_bin"].last()
        cycle_lengths = end_times - start_times

        end_times = end_times[tracklet_ap_bins.between(0.05, 0.95)]
        end_times = end_times - end_times.mean()
        tracklet_ap_bins = tracklet_ap_bins[tracklet_ap_bins.between(0.05, 0.95)]

        condition = condition_map[stems[k][:-6]]
        if condition == "trk":
            continue

        sns.lineplot(x=tracklet_ap_bins, y=end_times, errorbar=None, color=condition_main_colors[condition], alpha=0.3)

    plt.show()

In [ ]:
for k, df in enumerate(spots_dfs):
    df = df[df["AP"].between(0.05, 0.95)]
    df = df[df["cycle"] == 12]
    stem = stems[k]
    condition = condition_map[stem[:-6]]
    df_length = df["y"].max() - df["y"].min()
    sns.lineplot(df[df["pseudotime"].between(0.25, 0.35)], x="AP_bin", y="radius",
    color=condition_main_colors[condition], alpha=0.3, errorbar=None, label=f"{stems[k][:-6]}", legend=False)

In [ ]:
from tqdm import tqdm
from blender_tissue_cartography.mesh import ObjMesh
from natsort import natsorted

csvs_path = Path(r"C:\Tracking\BlastodermAnalysis\figures\output\all_meshes")

def hex2rgb(color):
    return int(color[1:3], 16) / 255, int(color[3:5], 16) / 255, int(color[5:7], 16) / 255


base_color = r"#e9d8a6"
r, g, b = hex2rgb(base_color)

colors = [hex2rgb(c) for c in
          ["#0a9396", "#ee9b00", "#ae2012"]]

track_id_colors = {}

def hex_to_rgb(hex_color):
    hex_color = hex_color.lstrip('#')
    return tuple(int(hex_color[i:i+2], 16) / 255 for i in (0, 2, 4))

df1 = pd.read_csv(csvs_path / "frame_26_colors.csv")
mesh1 = ObjMesh.read_obj(str(csvs_path / "frame_26.obj"))
positions = mesh1.vertices[np.unique(mesh1.faces.flatten())]

saved_positions = []

aps = positions[:, 1] - positions[:, 1].min()
aps = aps / aps.max()
np.random.seed(2)
for j, ap_bin in enumerate([(0.1, 0.15), (0.4, 0.6), (0.85, 0.9)]):
    ap_bin_mask = (aps > ap_bin[0]) & (aps < ap_bin[1]) & (positions[:, 0] > 100) & (positions[:, 2] > 120) & (positions[:, 2] < 150)
    ap_bin_indices = np.argwhere(ap_bin_mask)
    for idx in np.random.choice(ap_bin_indices.flatten(), 1, replace=False):
        saved_positions.append(aps[idx])
        track_id_colors[df1["track_id"].iloc[idx]] = colors[j]

for i, fp in tqdm(enumerate(natsorted(csvs_path.glob("*colors.csv")))):
    df = pd.read_csv(fp)


    new_rs = [track_id_colors.get(tid, (r, g, b))[0] for tid in df["track_id"].values]
    new_gs = [track_id_colors.get(tid, (r, g, b))[1] for tid in df["track_id"].values]
    new_bs = [track_id_colors.get(tid, (r, g, b))[2] for tid in df["track_id"].values]
    df["R"] = new_rs
    df["G"] = new_gs
    df["B"] = new_bs

    df.to_csv(str(fp)[:-4] + "_new2.csv", index=False)


In [ ]:
from tqdm import tqdm
from blender_tissue_cartography.mesh import ObjMesh
from natsort import natsorted

csvs_path = Path(r"C:\Tracking\BlastodermAnalysis\figures\output\all_meshes")

def hex2rgb(color):
    return int(color[1:3], 16) / 255, int(color[3:5], 16) / 255, int(color[5:7], 16) / 255


base_color = r"#e9d8a6"
r, g, b = hex2rgb(base_color)

colors = [hex2rgb(c) for c in
          ["#0a9396", "#ee9b00", "#ae2012"]]

track_id_colors = {}

def hex_to_rgb(hex_color):
    hex_color = hex_color.lstrip('#')
    return tuple(int(hex_color[i:i+2], 16) / 255 for i in (0, 2, 4))

df1 = pd.read_csv(csvs_path / "frame_26_colors.csv")
mesh1 = ObjMesh.read_obj(str(csvs_path / "frame_26.obj"))
positions = mesh1.vertices[np.unique(mesh1.faces.flatten())]

saved_positions = []

aps = positions[:, 1] - positions[:, 1].min()
aps = aps / aps.max()
np.random.seed(5)
for j, ap_bin in enumerate([(0.05, 0.25), (0.4, 0.6), (0.75, 0.95)]):
    ap_bin_mask = (aps > ap_bin[0]) & (aps < ap_bin[1])
    ap_bin_indices = np.argwhere(ap_bin_mask)
    for idx in ap_bin_indices.flatten():
        saved_positions.append(aps[idx])
        track_id_colors[df1["track_id"].iloc[idx]] = colors[j]

for i, fp in tqdm(enumerate(natsorted(csvs_path.glob("*colors.csv")))):
    df = pd.read_csv(fp)


    new_rs = [track_id_colors.get(tid, (r, g, b))[0] for tid in df["track_id"].values]
    new_gs = [track_id_colors.get(tid, (r, g, b))[1] for tid in df["track_id"].values]
    new_bs = [track_id_colors.get(tid, (r, g, b))[2] for tid in df["track_id"].values]
    df["R"] = new_rs
    df["G"] = new_gs
    df["B"] = new_bs

    df.to_csv(str(fp)[:-4] + "_new2.csv", index=False)


In [ ]:
k = 0
def hex2rgb(color):
    return int(color[1:3], 16) / 255, int(color[3:5], 16) / 255, int(color[5:7], 16) / 255


base_color = r"#e9d8a6"
r, g, b = hex2rgb(base_color)

colors = [hex2rgb(c) for c in
          ["#0a9396", "#ee9b00", "#ae2012"]]


fig, ax = plt.subplots(1, 1, figsize=(2.5, 2.2))
df = spots_dfs[k].copy()
first_frame = all_mmfs[stems[0]][0]
df = df[df["frame"] >= first_frame].copy()
df["initial_position"] = df.groupby("track_id")["AP"].transform("first")
df["displacement_from_start"] = (df["AP"] - df["initial_position"]) * 100
df["initial_cycle"] = df.groupby("track_id")["cycle"].transform("first")

df = df[df["frame"] == all_mmfs[stems[0]][-1]].copy()
df = df.query("initial_cycle == 10")
t = df.groupby("track_id")[["AP", "displacement_from_start", "initial_cycle"]].mean().reset_index()
print(t.index)

track_id_colors = {}
frame_df = df.query("frame == frame.min()")
frame_positions = frame_df.groupby("track_id")[["AP"]].mean()
for position, color in zip([0.15, 0.5, 0.9], colors):
    print(position)
    distances = (frame_positions["AP"] - position)**2

    print(distances.idxmin())
    print(t.set_index("track_id").loc[distances.idxmin(), "AP"])

    track_id_colors[distances.idxmin()] = color

cs = [track_id_colors.get(tid, hex2rgb("#D8D4D1")) for tid in t.track_id]
is_special = np.array([tid in track_id_colors.keys() for tid in t.track_id])

sns.scatterplot(t[~is_special], x="AP", y="displacement_from_start", color="#D8D4D1", edgecolor="k", legend=False, s=25, lw=1.0, alpha=0.7)
sns.scatterplot(t[is_special], x="AP", y="displacement_from_start", hue="track_id", palette=track_id_colors, edgecolor="k", legend=False, s=45)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.xlabel("AP Position")
plt.ylabel("Displacement along AP axis")
plt.axhline(0, linestyle="--", linewidth=2, color="k")
plt.savefig(save_path / f"{stems[k]}_dislacement.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
def get_plotting_df2(relative_densities_df):
    # remove most anterior and posterior positions (pole cells)
    plotting_df = relative_densities_df.query("positions > 0.01 and positions < 0.95").copy()

    # apply rolling mean for each density profile
    plotting_df["densities_local_smooth"] = (plotting_df.groupby(["cycle", "source"])["densities"]
                                                 .transform(lambda x: x.rolling(3, center=True, min_periods=1).mean()))

    # normalize by the average density in the middle region (0.4-0.6) for each cycle and source
    normalizing_constants = plotting_df[plotting_df["positions"].between(0.4, 0.6)].groupby(
        ["cycle", "source"])["densities_local_smooth"].mean().reset_index()

    # divide by the normalizing constant
    plotting_df["densities_local_smooth_normed"] = plotting_df.apply(
        lambda row: row["densities_local_smooth"] /
                    normalizing_constants[
                        (normalizing_constants["cycle"] == row["cycle"]) &
                        (normalizing_constants["source"] == row["source"])
                    ]["densities_local_smooth"].values[0], axis=1)

    return plotting_df

def plot_bar_plot_at_cycles(relative_densities_df, cycle, conditions, regions, ax, legend=False):
    plotting_df = get_plotting_df2(relative_densities_df)

    filtered_df = plotting_df.query("cycle == @cycle and condition in @conditions").copy()


    x = pd.cut(filtered_df.positions, bins=pd.IntervalIndex.from_tuples(
        [(0.0, 0.1), (0.4, 0.6), (0.85, 0.95)],
    ))

    x = x.cat.rename_categories(["Anterior", "Middle", "Posterior"])
    filtered_df["binned_position"] = x

    sns.barplot(filtered_df, x="cycle", y="densities_local_smooth", hue="binned_position", palette=colors, lw=1, edgecolor="k", legend=legend, ax=ax)

fig, ax = plt.subplots(1, 1, figsize=(3, 3))
plot_bar_plot_at_cycles(cycle_relative_densities, [10, 14], ["wt"], [], ax, legend=True)
plt.ylim(0.0, 1.2)
legend = ax.legend(loc="lower right")


# Create a new blank figure and copy the legend into it
legend_fig, legend_ax = plt.subplots(figsize=(2, 2))
legend_ax.axis("off")

# Copy legend handles and labels to the new figure
handles, labels = ax.get_legend_handles_labels()
legend_ax.legend(handles, labels, loc="center", frameon=True)

legend_fig.savefig(save_path / "legend.png", dpi=150, bbox_inches="tight")
plt.close(legend_fig)

ax.get_legend().remove()
ax.set_xlabel("Nuclear Cycle")
ax.set_ylabel("Relative Density")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.show()

# Glasbey blender

In [ ]:
import pandas as pd
from tqdm import tqdm
from collections import defaultdict
import napari


df = spots_dfs[0]


# viewer = napari.Viewer(ndisplay=3)
# colors = [cc.glasbey_cool[tid % 255] for tid in df["track_id"]]
# viewer.add_points(df[["frame", "z", "y", "x"]], face_color=colors, properties=df["track_id"].values, size=df["radius"]*3, border_color="k")
# napari.run()

mesh_path = save_path / "all_meshes"

def hex2rgb(color):
    return int(color[1:3], 16) / 255, int(color[3:5], 16) / 255, int(color[5:7], 16) / 255


base_color = r"#ffffff"
r, g, b = hex2rgb(base_color)

selected_track_ids = df.query("frame==frame.max() and x > 0").groupby("track_id")["frame"].count()
selected_track_ids = np.random.choice(selected_track_ids[selected_track_ids == 16].index, 10)
selected_track_ids = [31, 139, 175, 116]
track_id_colors = {}
for t in selected_track_ids:
    track_id_colors[t] = hex2rgb("#444444")

# viewer = napari.Viewer(ndisplay=3)
# colors = ["#444444" if tid in selected_track_ids else "#ffffff" for tid in df["track_id"]]
# viewer.add_points(df[["frame", "z", "y", "x"]], face_color=colors, properties=df["track_id"].values, size=df["radius"]*2)
# napari.run()

for i, frame in tqdm(enumerate(df["frame"].unique())):

    frame_df = df.query("frame == @frame")

    points = frame_df[["z", "y", "x"]].values
    mesh = dnt.mesh_from_points(points)

    mesh.write_obj(mesh_path / f"frame_{frame}.obj")
    track_ids = frame_df["track_id"].values

    colors = [track_id_colors.get(tid, hex2rgb(base_color)) for tid in track_ids]

    valid = pd.Series(np.arange(len(points))).isin(np.unique(mesh.faces))
    blender_save = pd.DataFrame(np.array(colors)[valid], columns=["R", "G", "B"])
    blender_save["track_id"] = track_ids[valid]

    blender_save.to_csv(mesh_path / f"frame_{frame}_colors.csv", index=False)

# continuous / rainbow blender

In [ ]:
# from tqdm import tqdm
# from blender_tissue_cartography.mesh import ObjMesh
# from natsort import natsorted
#
# csvs_path = Path(r"C:\Tracking\BlastodermAnalysis\figures\output\all_meshes")
#
# n_colors = 7
# colors = sns.color_palette("Spectral", as_cmap=True)
#
# track_id_colors = {}
#
# def hex_to_rgb(hex_color):
#     hex_color = hex_color.lstrip('#')
#     return tuple(int(hex_color[i:i+2], 16) / 255 for i in (0, 2, 4))
#
# df1 = pd.read_csv(csvs_path / "frame_50_colors.csv")
# print(df1["track_id"].nunique())
# mesh1 = ObjMesh.read_obj(str(csvs_path / "frame_50.obj"))
# print(np.unique(mesh1.faces.flatten()))
# print(mesh1.vertices)
#
# vertices = mesh1.vertices[np.unique(mesh1.faces.flatten())]
#
# positions = vertices
# print(len(positions), len(df1))
# aps = positions[:, 1] - positions[:, 1].min()
# aps = aps / aps.max()
# for idx, ap in enumerate(aps[:len(df1)]):
#     track_id_colors[df1["track_id"].loc[idx]] = colors(ap)
#
# for i, fp in tqdm(enumerate(natsorted(csvs_path.glob("*colors.csv")))):
#     df = pd.read_csv(fp)
#
#
#     new_rs = [track_id_colors.get(tid, (0.4, 0.4, 0.4))[0] for tid in df["track_id"].values]
#     new_gs = [track_id_colors.get(tid, (0.4, 0.4, 0.4))[1] for tid in df["track_id"].values]
#     new_bs = [track_id_colors.get(tid, (0.4, 0.4, 0.4))[2] for tid in df["track_id"].values]
#     df["R"] = new_rs
#     df["G"] = new_gs
#     df["B"] = new_bs
#
#     df.to_csv(str(fp)[:-4] + "_new.csv", index=False)
